# 02 天衍接口、精确概率与有限读数

本节属于课程进阶模块。代码、教学构造、历史实测、待执行实验分别标注。云执行默认关闭；真实金融数据通过仓库外的私有快照接入。原始行情、逐样本标签、角度、核矩阵、参数与预测不得存入公开Notebook输出。

**来源**：[L001 ICLR正式版](https://proceedings.iclr.cc/paper_files/paper/2024/file/4d9944ab3330fe6af8efb9260aa9f307-Paper-Conference.pdf)、[本课程协议](../../docs/天衍云端复现协议.md)。署名：Julius Zhou、AUFE、Cong Lei。代码MIT，原创教学文字与图表CC BY 4.0。


In [1]:
from pathlib import Path
import sys, os, json
import numpy as np
import pandas as pd
COURSE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "课程大纲.md").exists())
MODULE = COURSE / "experiments/qukernet_finance"
sys.path.insert(0, str(MODULE))
PUBLIC = COURSE / "reports/qukernet_cloud_v2"
ENABLE_CLOUD = False
print("默认云执行关闭；当前Notebook不读取凭据、不提交任务。")


默认云执行关闭；当前Notebook不读取凭据、不提交任务。


## 三种计算位置
SDK在无影本地模拟、远程MCP服务进程内模拟、天衍云队列任务是三种位置。调用名中出现“simulate”不证明使用了云后端。`sa`指定输出态的接口线索尚不等于已确认精确概率契约；`sw`有限读数必须保留原始`resultStatus`。

本节先做纯离线条件检查。真正认证与提交只在独立证据满足后，通过Linux客户端显式开启。不得用提交前后余额比较替代计费规则。

In [2]:
from cloud_v2.preflight import validate_evidence, PreflightBlocked
try:
    validate_evidence({}, "0"*64)
except PreflightBlocked as exc:
    print("预期阻断:", str(exc))
else:
    raise AssertionError("缺失费用证据却放行")
print("实际云调用=0")

预期阻断: Evidence scope mismatch: account_scope_sha256
实际云调用=0


## 位序、原始读数与哈希
输入QCIS使用显式Q1…Qq，返回测量列可能采用不同顺序，按列标签对应。每条线路读数数目须等于请求shots；零串频率只依赖是否全部为0，但其他概率表对账仍须核位序。返回QCIS可用时检查输入哈希。高层SDK可能由`probability × shots`构造counts，这类counts不能冒充逐次采样。

每批最多20条，提交前预留预算，未知任务不重发。全局72小时、100亿shots、8000万线路，以及预检50条/2万shots共同约束。

In [3]:
import hashlib
from cloud_v2.preflight import normalize_qcis
text="RY Q1 0.3\nH Q2\nCZ Q1 Q2\nM Q1\nM Q2"
digest=hashlib.sha256(normalize_qcis(text).encode()).hexdigest()
print("解析示例输入SHA256:",digest)
print("这只是哈希教学，没有提交线路。")

解析示例输入SHA256: 190d3acf607ccfac6045f11949af8c8ecce2c30a6dec25597ecaa2d825a6221c
这只是哈希教学，没有提交线路。


## 显式运行入口
在无影隔离环境中使用`python -m cloud_v2.client_entry --execute`，同时提供私有证据文件、固定账户账本位置、窗口、输出与阶段参数。入口先检查证据，再接受隐藏输入或进程环境变量中的凭据；凭据不得出现在Notebook单元格。

**验收与作业**：解释空结果列表为何不能写作“正在排队”；列出“提交超时但后台可能接受”应保留哪些证据。测试SDK自动重试、部分任务ID、未知状态阻止新批次三个反例。